# KinoCompass – итоговый тест: EASE

**Тест – один раз.** EASE с λ = 5000 (выбран на валидации) переобучается на всём до 14.10.2022 и проверяется на последнем годе. Модель не меняется.

На тесте 22 309 кандидатов: матрица «фильм × фильм» в float64 – 4 ГБ. Обращение – на GPU через разложение Холецкого полосами по 2048 столбцов (в памяти: разложение 4 ГБ + итоговая матрица 2 ГБ + полоса), а **до расчёта** программа проверяет свободную видеопамять и сразу останавливается с понятным сообщением, если её не хватает.

Состав (`python evaluate.py final_ease`): популярные за последний год (для парных сравнений) и EASE λ = 5000.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**, затем «Среда выполнения» → «Выполнить все». В конце скачается архив `final_ease_test.zip` – его нужно положить в `model/reports/`.

Время на T4 (оценка): загрузка набора и сценарии – около 3 минут, XᵀX и обращение – 2–5 минут, проверка – около 2 минут; всего около 10 минут.

In [ ]:
# итоговый тест – один раз (модели выбраны на валидации, шаг 6в)
PERIOD = "test"

In [ ]:
import torch

assert torch.cuda.is_available(), "нет GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

In [ ]:
# из /content: при повторном запуске в том же сеансе текущая папка – внутри удаляемого клона
%cd /content
!rm -rf /content/KinoCompass && git clone -q --depth 1 https://github.com/AlexTrav/KinoCompass.git /content/KinoCompass
!git -C /content/KinoCompass log --oneline -1
# те же версии, что в образе Docker (torch в Colab уже есть)
!pip install -q requests==2.34.2 pandas==3.1.0 pyarrow==26.0.1 scipy==1.18.1

In [ ]:
%cd /content/KinoCompass/model
!PYTHONPATH=/content/KinoCompass/data:/content/KinoCompass/research PERIOD={PERIOD} python -W ignore evaluate.py final_ease

In [ ]:
from google.colab import files

!cd /content/KinoCompass/model && zip -q -r /content/final_ease_{PERIOD}.zip reports/final_ease && ls -la reports/final_ease
files.download(f"/content/final_ease_{PERIOD}.zip")